In [1]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║         PDF → Markdown  |  Powered by Baidu Unlimited-OCR       ║
# ║         Google Colab Compatible  •  GPU Runtime Required         ║
# ║  GitHub: https://github.com/baidu/Unlimited-OCR                  ║
# ╚══════════════════════════════════════════════════════════════════╝
#
## ⚠️ Important

# Before running the script, go to Colab Runtime → Change runtime type and select T4 GPU or A100.

# How to Use

# Follow these 3 simple steps:

# 1. Upload the script to Google Colab or copy the code into Colab cells.
# 2. Set the PDF_PATH variable to the path of your PDF file.
# 3. Run all cells. The output .md file will be generated automatically and downloaded.

# ══════════════════════════════════════════════════════════════════

In [2]:
# ─────────────────────────────────────────────────────────────────
# CELL 1 ▸ GPU CHECK
# ─────────────────────────────────────────────────────────────────

import subprocess, sys

def check_gpu():
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=10
        )
        if result.returncode == 0:
            gpu_info = result.stdout.strip()
            print(f"✅ GPU detected: {gpu_info}")
            return True
        else:
            print("❌ GPU Not Detected! Please open Runtime → Change runtime type → Hardware accelerator, then select T4 GPU and try again.")
            return False
    except Exception as e:
        print(f"❌ nvidia-smi error: {e}")
        return False

gpu_ok = check_gpu()
if not gpu_ok:
    raise SystemExit("GPU Required! Please change the runtime type to a GPU-enabled runtime and run the process again.")

✅ GPU detected: Tesla T4, 15360 MiB


In [3]:
# ─────────────────────────────────────────────────────────────────
# CELL 2 ▸ INSTALL DEPENDENCIES
# ─────────────────────────────────────────────────────────────────
#
# NOTE: This cell takes ~2–3 minutes to complete. Output is suppressed, so please wait.
#
# If you want to run it manually, paste the following in the terminal:
# pymupdf==1.27.2.2 psutil==7.2.2 -q
#
# NOTE: Pillow 12.1.1 (and 12.0.0) has a known broken wheel issue.
# PIL/ImageText.py imports `_Ink`, which does not exist in PIL/_typing.py,
# causing torchvision imports to fail.
#
# Therefore, Pillow is pinned to 11.3.0 (the last stable 11.x release),
# which does not contain this buggy module.
#
# ─────────────────────────────────────────────────────────────────

print("📦 Installing dependencies... (2-3 min)")

packages = [
    "torch==2.10.0",
    "torchvision==0.25.0",
    "transformers==4.57.1",
    "Pillow==11.3.0",
    "einops==0.8.2",
    "addict==2.4.0",
    "easydict==1.13",
    "pymupdf==1.27.2.2",
    "psutil==7.2.2",
]

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet"] + packages,
    capture_output=True, text=True
)

if result.returncode == 0:
    print("✅ All packages installed successfully!")
else:
    print("⚠️  Some packages may have warnings:")
    print(result.stderr[-1000:] if result.stderr else "")
    print("Continuing anyway...")

print(
    "\n⚠️ IMPORTANT: If you previously ran the old Pillow==12.1.1 version and imported PIL/torch, restart the Colab session (Runtime → Restart session) and run again from Cell 1; otherwise, the old broken PIL module may remain cached in memory."
)

📦 Installing dependencies... (2-3 min)
✅ All packages installed successfully!

⚠️ IMPORTANT: If you previously ran the old Pillow==12.1.1 version and imported PIL/torch, restart the Colab session (Runtime → Restart session) and run again from Cell 1; otherwise, the old broken PIL module may remain cached in memory.


In [4]:
# ─────────────────────────────────────────────────────────────────
# CELL 3 ▸ IMPORTS
# ─────────────────────────────────────────────────────────────────

import os
import re
import glob
import tempfile
import shutil
from pathlib import Path

import torch
import fitz  # PyMuPDF

print("✅ Core imports done.")
print(f"   PyTorch version : {torch.__version__}")
print(f"   CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU             : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"   VRAM            : {vram:.1f} GB")

✅ Core imports done.
   PyTorch version : 2.10.0+cu128
   CUDA available  : True
   GPU             : Tesla T4
   VRAM            : 15.6 GB


In [5]:
# ─────────────────────────────────────────────────────────────────
# CELL 4 ▸ LOAD MODEL  (The first run may take 5–15 minutes because the ~7 GB model needs to be downloaded.)
# ─────────────────────────────────────────────────────────────────

from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "baidu/Unlimited-OCR"

print(f"\n🤖 Loading model: {MODEL_NAME}")
print("   (First run: ~7 GB download — please be patient)\n")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)
print("✅ Tokenizer loaded.")

model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_safetensors=True,
    torch_dtype=torch.bfloat16,
)
model = model.eval().cuda()
print("✅ Model loaded & moved to GPU!\n")


🤖 Loading model: baidu/Unlimited-OCR
   (First run: ~7 GB download — please be patient)



/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/baidu/Unlimited-OCR:
- modeling_deepseekv2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/baidu/Unlimited-OCR:
- modeling_unlimitedocr.py
- conversation.py
- deepencoder.py
- configuration_deepseek_v2.py
- modeling_deepseekv2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-000001.safetensors:   0%|          | 0.00/6.67G [00:00<?, ?B/s]

Some weights of UnlimitedOCRForCausalLM were not initialized from the model checkpoint at baidu/Unlimited-OCR and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model loaded & moved to GPU!



In [8]:
# ─────────────────────────────────────────────────────────────────
# CELL 5 ▸ HELPER FUNCTIONS (WITH TRUE RAW MODEL GENERATION)
# ─────────────────────────────────────────────────────────────────

import torch
from PIL import Image

def pdf_to_images(pdf_path: str, dpi: int = 300) -> list:
    """
    Convert each page of the PDF to a PNG image at the specified DPI.
    """
    doc = fitz.open(pdf_path)
    total_pages = len(doc)
    tmp_dir = tempfile.mkdtemp(prefix="pdf_pages_")
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    paths = []

    print(f"   📄 PDF pages     : {total_pages}")
    print(f"   🖼️  DPI           : {dpi}")
    print(f"   📁 Temp dir      : {tmp_dir}")
    print()

    for i, page in enumerate(doc):
        out_path = os.path.join(tmp_dir, f"page_{i + 1:04d}.png")
        page.get_pixmap(matrix=mat).save(out_path)
        paths.append(out_path)
        done = int(30 * (i + 1) / total_pages)
        bar = "█" * done + "░" * (30 - done)
        print(f"\r   [{bar}] {i+1}/{total_pages}", end="", flush=True)

    doc.close()
    print(f"\n   ✅ All {total_pages} pages converted to images.\n")
    return paths

_DET_RE = re.compile(
    r"<\|det\|>([^<\s]+)(?:\s*\[[^\]]*\])?\s*<\|/det\|>(.*)",
    re.DOTALL,
)

def remove_det(raw: str) -> str:
    """
    Strip <|det|>type [bbox]<|/det|> markers.
    Ignore all images and convert only the existing text into Markdown.
    """
    blocks = []
    cur = None
    for line in raw.splitlines():
        line = line.rstrip()
        if not line:
            continue
        m = _DET_RE.match(line)
        if m:
            category = m.group(1).strip()
            content = m.group(2).strip()
            if category == "image":
                continue
            if cur is not None:
                blocks.append(cur)
            cur = [content] if content else []
            continue
        if cur is None:
            cur = []
        cur.append(line)
    if cur is not None:
        blocks.append(cur)
    return "\n\n".join("\n".join(b) for b in blocks).strip()


_IMG_MD_RE = re.compile(r"!\[[^\]]*\]\([^\)]*\)")

def strip_broken_images(text: str) -> str:
    lines = text.splitlines()
    out = []
    for line in lines:
        if _IMG_MD_RE.search(line.strip()):
            out.append("*[Image on this page — not extracted]*")
        else:
            out.append(line)
    return "\n".join(out)


_BULLET_PREFIX_RE = re.compile(r"^[\s○●•\-\*]+")

def collapse_repetition(text: str, prefix_len: int = 10, max_repeat: int = 2) -> str:
    lines = text.split("\n")
    out = []
    i, n = 0, len(lines)
    while i < n:
        norm = _BULLET_PREFIX_RE.sub("", lines[i]).strip().lower()
        prefix = norm[:prefix_len]

        if not prefix:
            out.append(lines[i])
            i += 1
            continue

        j = i + 1
        while j < n:
            nxt = _BULLET_PREFIX_RE.sub("", lines[j]).strip().lower()
            if nxt[:prefix_len] == prefix:
                j += 1
            else:
                break

        run_len = j - i
        if run_len > max_repeat:
            out.extend(lines[i:i + max_repeat])
            out.append(f"*…[{run_len - max_repeat} repeated/degenerate lines trimmed]…*")
            i = j
        else:
            out.append(lines[i])
            i += 1

    return "\n".join(out)


def combine_outputs(outputs: list[str]) -> str:
    pages_text = []
    for raw in outputs:
        try:
            cleaned = raw
            cleaned = remove_det(raw)
            cleaned = strip_broken_images(cleaned)
            cleaned = collapse_repetition(cleaned)
            if cleaned.strip():
                pages_text.append(cleaned)
        except Exception as e:
            pass
    return "\n\n".join(pages_text)

In [17]:
# ─────────────────────────────────────────────────────────────────
# CELL 6 ▸ CONFIGURATION  ← ★ ONLY CHANGE THIS ★
# ─────────────────────────────────────────────────────────────────

# ★ Enter the path to your PDF or image file here (.pdf, .jpg, .png, .bmp, .tiff, .webp):
PDF_PATH = "/content/a-functional-description-of-texs-formula-layout-1-9.pdf"

# Optional settings:
DPI = 300          # Image resolution (higher = better quality, slower)
                   # Recommended: 200 (fast) | 300 (balanced) | 400 (high quality)

BATCH_SIZE = 5     # How many pages should be given to the GPU at once.
                   # On a T4 (15GB), giving 45+ pages at once causes CUDA OOM.
                    # If OOM still occurs, reduce this to 3 or 2.
                   # If you have an A100 (40GB), you can increase it to 10-15.

# ─────────────────────────────────────────────────────────────────

In [18]:
import io
from contextlib import redirect_stdout

def process_pdf_to_markdown(pdf_path: str, dpi: int = 300, batch_size: int = None) -> str:
    """
    Take the PDF file and create a .md file with the same name.
    Now each page is processed SEPARATELY, in 'gundam' (crop) mode.
    We directly capture the return value of model.infer or its terminal stdout if it prints directly.
    """
    pdf_path = Path(pdf_path).resolve()

    if not pdf_path.exists():
        raise FileNotFoundError(f"❌ PDF file was not found: {pdf_path}")
    if pdf_path.suffix.lower() != ".pdf":
        raise ValueError(f"❌ File is not a PDF: {pdf_path.name}")

    md_path = pdf_path.with_suffix(".md")
    md_raw_path = pdf_path.with_name(f"{pdf_path.stem}_raw.txt")

    print("=" * 60)
    print("🚀 Unlimited-OCR PDF → Markdown Converter (gundam / per-page)")
    print("=" * 60)
    print(f"   📄 Input PDF  : {pdf_path.name}")
    print(f"   📝 Output MD  : {md_path.name}")
    print(f"   📝 True Raw   : {md_raw_path.name}")
    print()

    print("📸 Step 1/3 — Converting PDF pages to images...")
    image_files = pdf_to_images(str(pdf_path), dpi=dpi)
    total = len(image_files)

    ocr_root_dir = tempfile.mkdtemp(prefix="ocr_results_")
    print(f"🔍 Step 2/3 — Running OCR on {total} pages (gundam mode, 1-by-1)...")
    print(f"   (Capturing terminal stdout of model.infer to get raw output)\n")
    isErrored = False
    page_dirs = []
    raw_responses = []

    for idx, img_path in enumerate(image_files, start=1):
        page_dir = os.path.join(ocr_root_dir, f"page_{idx:04d}")
        os.makedirs(page_dir, exist_ok=True)

        print(f"   ▶️  Page {idx}/{total} ...", end=" ")
        captured_output = io.StringIO()
        try:
            if isErrored:
                raise torch.cuda.OutOfMemoryError

            # Capture standard output printed by model.infer
            with redirect_stdout(captured_output):
                model.infer(
                    tokenizer,
                    prompt='<image>document parsing.',
                    image_file=img_path,
                    output_path=page_dir,
                    base_size=1024, image_size=640, crop_mode=True,
                    max_length=32768,
                    no_repeat_ngram_size=35, ngram_window=128,
                    save_results=True,
                )
            raw_response = captured_output.getvalue()
            print("✅")
        except torch.cuda.OutOfMemoryError:
            if not isErrored:
                print("❌ OOM — retrying in 'base' mode (fallback)")
            torch.cuda.empty_cache()
            isErrored = True

            captured_output = io.StringIO()
            with redirect_stdout(captured_output):
                model.infer(
                    tokenizer,
                    prompt='<image>document parsing.',
                    image_file=img_path,
                    output_path=page_dir,
                    base_size=1024, image_size=1024, crop_mode=False,
                    max_length=32768,
                    no_repeat_ngram_size=35, ngram_window=128,
                    save_results=True,
                )
            raw_response = captured_output.getvalue()
        finally:
            torch.cuda.empty_cache()

        page_dirs.append(page_dir)
        raw_responses.append(raw_response if raw_response else "")

    print("\n   ✅ OCR complete!")
    print(f"   📁 Raw output saved to: {ocr_root_dir}")

    print("\n📖 Step 3/3 — Reading & combining OCR output...")

    combined_text = combine_outputs(raw_responses)

    # Combine the true captured raw output strings directly
    true_raw_combined = "\n\n".join(raw_responses)

    final_md = f"""# {pdf_path.stem}

> *Generated by [Unlimited-OCR](https://github.com/baidu/Unlimited-OCR) (Baidu)*
> *Source PDF: `{pdf_path.name}`*

---

{combined_text}
"""

    with open(str(md_path), "w", encoding="utf-8") as f:
        f.write(final_md)

    # Save the genuine raw untouched output containing <|det|> tags
    with open(str(md_raw_path), "w", encoding="utf-8") as f:
        f.write(true_raw_combined)

    char_count = len(combined_text)
    word_count = len(combined_text.split())
    md_size_kb = md_path.stat().st_size / 1024

    print()
    print("=" * 60)
    print("✅ SUCCESS!")
    print("=" * 60)
    print(f"   📝 Output file : {md_path}")
    print(f"   📝 True Raw file: {md_raw_path}")
    print(f"   📊 Characters  : {char_count:,}")
    print(f"   📊 Words       : {word_count:,}")
    print(f"   💾 File size   : {md_size_kb:.1f} KB")

    try:
        img_tmp_dir = os.path.dirname(image_files[0])
        shutil.rmtree(img_tmp_dir, ignore_errors=True)
        print(f"   🧹 Temp images cleaned up.")
    except Exception:
        pass

    return str(md_path)

In [19]:
# ─────────────────────────────────────────────────────────────────
# CELL 8 ▸ RUN!
# ─────────────────────────────────────────────────────────────────

output_md_path = process_pdf_to_markdown(PDF_PATH, dpi=DPI, batch_size=BATCH_SIZE)

🚀 Unlimited-OCR PDF → Markdown Converter (gundam / per-page)
   📄 Input PDF  : a-functional-description-of-texs-formula-layout-1-9.pdf
   📝 Output MD  : a-functional-description-of-texs-formula-layout-1-9.md
   📝 True Raw   : a-functional-description-of-texs-formula-layout-1-9_raw.txt

📸 Step 1/3 — Converting PDF pages to images...
   📄 PDF pages     : 9
   🖼️  DPI           : 300
   📁 Temp dir      : /tmp/pdf_pages_ocwjbxgw

   [██████████████████████████████] 9/9
   ✅ All 9 pages converted to images.

🔍 Step 2/3 — Running OCR on 9 pages (gundam mode, 1-by-1)...
   (Capturing terminal stdout of model.infer to get raw output)

   ▶️  Page 1/9 ... 

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


❌ OOM — retrying in 'base' mode (fallback)


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
image: 0it [00:00, ?it/s]
other: 100%|██████████| 18/18 [00:00<00:00, 93206.76it/s]
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


   ▶️  Page 2/9 ... 

...

   ▶️  Page 8/9 ... 

other: 100%|██████████| 12/12 [00:00<00:00, 95144.89it/s]
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


   ▶️  Page 9/9 ... 

other: 100%|██████████| 10/10 [00:00<00:00, 83718.64it/s]


   ✅ OCR complete!
   📁 Raw output saved to: /tmp/ocr_results_xiwx30fv

📖 Step 3/3 — Reading & combining OCR output...

✅ SUCCESS!
   📝 Output file : /content/a-functional-description-of-texs-formula-layout-1-9.md
   📝 True Raw file: /content/a-functional-description-of-texs-formula-layout-1-9_raw.txt
   📊 Characters  : 26,601
   📊 Words       : 4,214
   💾 File size   : 26.2 KB
   🧹 Temp images cleaned up.


In [20]:
# ─────────────────────────────────────────────────────────────────
# CELL 9 ▸ PREVIEW + DOWNLOAD (WITH RAW OUTPUT)
# ─────────────────────────────────────────────────────────────────

# Define the raw output path (which is saved as _raw.txt by Cell 7)
output_raw_txt_path = output_md_path.replace(".md", "_raw.txt")

# Preview the first 1000 characters of the processed output
print("═" * 60)
print("📖 MARKDOWN PREVIEW (Processed - first 1000 chars)")
print("═" * 60)
with open(output_md_path, "r", encoding="utf-8") as f:
    preview = f.read(1000)
print(preview)
if len(open(output_md_path).read()) > 1000:
    print("\n... [truncated] ...")

# Preview the first 1000 characters of the true raw output
print("\n" + "═" * 60)
print("📄 RAW OCR PREVIEW (With untouched <|det|> tags - first 1000 chars)")
print("═" * 60)
if os.path.exists(output_raw_txt_path):
    with open(output_raw_txt_path, "r", encoding="utf-8") as f:
        raw_preview = f.read(1000)
    print(raw_preview)
    if len(open(output_raw_txt_path).read()) > 1000:
        print("\n... [truncated] ...")
else:
    print(f"⚠️ Raw file not found at: {output_raw_txt_path}")

# Auto-download both files in Colab
try:
    from google.colab import files
    print(f"\n⬇️  Downloading Processed: {Path(output_md_path).name}")
    files.download(output_md_path)

    if os.path.exists(output_raw_txt_path):
        print(f"⬇️  Downloading Raw: {Path(output_raw_txt_path).name}")
        files.download(output_raw_txt_path)

    print("✅ Download started!")
except ImportError:
    # It is running outside Colab
    print(f"\n📁 Processed file saved at: {output_md_path}")
    if os.path.exists(output_raw_txt_path):
        print(f"📁 Raw file saved at: {output_raw_txt_path}")
except Exception as e:
    print(f"\n⚠️  Auto-download failed: {e}")
    print(f"   Manually download them from the Colab left sidebar.")

════════════════════════════════════════════════════════════
📖 MARKDOWN PREVIEW (Processed - first 1000 chars)
════════════════════════════════════════════════════════════
# a-functional-description-of-texs-formula-layout-1-9

> *Generated by [Unlimited-OCR](https://github.com/baidu/Unlimited-OCR) (Baidu)*
> *Source PDF: `a-functional-description-of-texs-formula-layout-1-9.pdf`*

---

J. Functional Programming 7 (5): 451–485, September 1997. Printed in the United Kingdom

© 1997 Cambridge University Press

451

A functional description of TEX's formula layout

REINHOLD HECKMANN and REINHARD WILHELM

Fachbereich Informatik, Universität des Saarlandes

Saurbrücken, Germany

(e-mail: {heckmann, wilhelm}@cs.uni-sb.de)

Abstract

While the quality of the results of \(\mathrm{TEX}\) 's mathematical formula layout algorithm is convincing, its original description is hard to understand since it is presented as an imperative program with complex control flow and destructive manipulations of the

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Downloading Raw: a-functional-description-of-texs-formula-layout-1-9_raw.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started!


In [21]:
# ─────────────────────────────────────────────────────────────────
# CELL 10 ▸ BATCH MODE (optional — multiple PDFs at once)
# ─────────────────────────────────────────────────────────────────

def process_multiple_pdfs(pdf_folder: str, dpi: int = 300, batch_size: int = 5):
    """
    Process all PDF files in a folder.

    Usage:
        process_multiple_pdfs("/content/my_pdfs/")
    """
    pdf_files = list(Path(pdf_folder).glob("*.pdf"))
    if not pdf_files:
        print(f"❌ No PDF was found: {pdf_folder}")
        return

    print(f"📚 {len(pdf_files)} PDFs found. Processing...")
    results = []

    for i, pdf in enumerate(pdf_files, 1):
        print(f"\n[{i}/{len(pdf_files)}] Processing: {pdf.name}")
        try:
            md = process_pdf_to_markdown(str(pdf), dpi=dpi, batch_size=batch_size)
            results.append({"pdf": pdf.name, "md": md, "status": "✅ Success"})
        except Exception as e:
            print(f"   ❌ Failed: {e}")
            results.append({"pdf": pdf.name, "md": None, "status": f"❌ {e}"})

    print("\n" + "=" * 60)
    print("📊 BATCH SUMMARY")
    print("=" * 60)
    for r in results:
        print(f"  {r['status']}  {r['pdf']}")
    print("=" * 60)

    # Batch download (Colab only)
    try:
        from google.colab import files
        for r in results:
            if r["md"]:
                files.download(r["md"])
    except ImportError:
        pass

    return results

# If you want to use batch mode:
# process_multiple_pdfs("/content/pdfs/")